# 🔍 Caderno 02: Redução de Dimensionalidade (PCA) e Classificação com KNN
**Projeto:** Detecção de Adulteração em Fertilizantes NPK via LIBS  
**Instituição:** Embrapa Instrumentação  

---

### Objetivos deste Caderno:
1. **Redução de Dimensionalidade via PCA (Análise de Componentes Principais)** nos canais VIS e UV.
2. **Interpretação Quimiométrica dos Loadings:** Mapeamento automático dos maiores pesos espectrais com a biblioteca atômica NIST (linhas de Ca II, Ca I, K, Mg, Fe, P).
3. **Classificação Supervisionada com KNN (K-Nearest Neighbors)** no espaço latente de scores.
4. **Validação Rigorosa contra Vazamento de Dados (Data Leakage):**
   - Comparação entre split aleatório padrão e **Leave-Group-Out CV (LOGO-CV)** agrupado por pastilha (`Amostra_ID`).
   - **Validação Cruzada de Monte Carlo** com subamostragem balanceada.

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.models import (
    executar_pca,
    identificar_top_loadings,
    avaliar_knn_leave_group_out,
    validacao_monte_carlo_knn
)
from src.features import (
    LINHAS_EMISSAO_VIS,
    CORES_ELEMENTOS_VIS,
    LINHAS_EMISSAO_UV,
    CORES_ELEMENTOS_UV
)
from src.visualization import (
    plotar_pca_scores,
    plotar_loadings_com_linhas_nist
)

print("Módulos carregados com sucesso!")

## 1. Carregamento dos Dados Pré-processados

In [ ]:
pasta_proc = BASE_DIR / "data" / "processed"
caminho_vis = pasta_proc / "df_limpo_vis.parquet"
caminho_uv = pasta_proc / "df_limpo_uv.parquet"

df_vis = pd.read_parquet(caminho_vis) if caminho_vis.exists() else None
df_uv = pd.read_parquet(caminho_uv) if caminho_uv.exists() else None

if df_vis is not None:
    print(f"Dados VIS carregados: {df_vis.shape}")
    print(df_vis.groupby(['Teor_Adulteracao', 'Amostra_ID']).size().unstack(fill_value=0))
else:
    print("Aviso: execute primeiro o Caderno 01 para gerar df_limpo_vis.parquet.")

## 2. PCA no Canal VIS
Executamos a PCA sobre os comprimentos de onda na faixa de 273 nm a 780 nm.

In [ ]:
if df_vis is not None:
    # Exclusão opcional de pastilhas de validação externa ou teores específicos
    df_analise = df_vis[df_vis['Amostra_ID'] != 'i_'].copy()

    pca_vis, X_pca_vis, cols_wl_vis, df_scores_vis = executar_pca(
        df_analise,
        wl_min=273.0,
        wl_max=780.0,
        n_components=5,
        scaler=False
    )

    print("Variância Explicada por Componente:")
    for i, var in enumerate(pca_vis.explained_variance_ratio_):
        print(f"  PC{i+1}: {var*100:.2f}% (Acumulado: {np.sum(pca_vis.explained_variance_ratio_[:i+1])*100:.2f}%)")

    # Gráfico de Scores PC1 vs PC2
    fig = plotar_pca_scores(df_scores_vis, pc_x="PC1", pc_y="PC2", titulo="Scores PCA - Canal VIS")
    plt.show()

## 3. Interpretação dos Loadings e Identificação de Linhas NIST
Analisamos quais comprimentos de onda são responsáveis pela separação observada na PC1 e PC2.

In [ ]:
if df_vis is not None:
    wls_vis = np.array([float(c) for c in cols_wl_vis])

    # Plot dos Loadings da PC1 e PC2
    fig1 = plotar_loadings_com_linhas_nist(
        wls_vis, pca_vis.components_[0], "PC1",
        pca_vis.explained_variance_ratio_[0]*100,
        LINHAS_EMISSAO_VIS, CORES_ELEMENTOS_VIS
    )
    plt.show()

    fig2 = plotar_loadings_com_linhas_nist(
        wls_vis, pca_vis.components_[1], "PC2",
        pca_vis.explained_variance_ratio_[1]*100,
        LINHAS_EMISSAO_VIS, CORES_ELEMENTOS_VIS
    )
    plt.show()

    # Tabela com os Top-10 picos identificados
    df_top_vis = identificar_top_loadings(
        pca_vis, cols_wl_vis, n_pcs=2, top_n=10,
        linhas_referencia=LINHAS_EMISSAO_VIS
    )
    print("Top Loadings Identificados:")
    display(df_top_vis) if 'display' in dir() else print(df_top_vis.to_string())

## 4. Classificação com KNN: Teste contra Vazamento de Dados (Data Leakage)
Em LIBS, múltiplos disparos são realizados sobre uma mesma pastilha.  
Se disparos da mesma pastilha caírem simultaneamente no treino e no teste, a acurácia é inflada artificialmente.  
Aqui testamos a validação estrita **Leave-Group-Out**, garantindo que pastilhas inteiras fiquem restritas ao teste.

In [ ]:
if df_vis is not None:
    # Definir grupos de pastilhas para treino e teste
    grupos = df_scores_vis['Amostra_ID'].values
    y = df_scores_vis['Teor_Adulteracao'].values
    X = X_pca_vis[:, :3] # Utilizando as primeiras 3 PCs

    amostras_unicas = list(np.unique(grupos))
    print(f"Pastilhas disponíveis: {amostras_unicas}")

    # Exemplo: treino nas primeiras pastilhas, teste nas últimas
    treino_ids = amostras_unicas[:-2]
    teste_ids = amostras_unicas[-2:]

    resultado_lgo = avaliar_knn_leave_group_out(
        X, y, grupos,
        ids_treino=treino_ids,
        ids_teste=teste_ids,
        n_neighbors=5
    )

    print(f"Treino IDs: {treino_ids} ({resultado_lgo['n_treino']} disparos)")
    print(f"Teste IDs:  {teste_ids} ({resultado_lgo['n_teste']} disparos)")
    print(f"Acurácia Leave-Group-Out: {resultado_lgo['acuracia']*100:.2f}%")

    plt.figure(figsize=(6, 5))
    sns.heatmap(resultado_lgo['matriz_confusao'], annot=True, fmt='d', cmap='Blues')
    plt.title("Matriz de Confusão - Leave Group Out CV")
    plt.xlabel("Predição")
    plt.ylabel("Real")
    plt.show()

## 5. Validação Cruzada de Monte Carlo (100 Iterações)
Subamostragem balanceada com 100 iterações para obter o intervalo de confiança empírico de 95% da acurácia.

In [ ]:
if df_vis is not None:
    res_mc = validacao_monte_carlo_knn(
        df_scores_vis,
        cols_features=['PC1', 'PC2', 'PC3'],
        col_alvo='Teor_Adulteracao',
        n_iteracoes=100,
        n_neighbors=5
    )

    print(f"Acurácia Média Monte Carlo: {res_mc['media_acuracia']*100:.2f}% ± {res_mc['desvio_acuracia']*100:.2f}%")
    print(f"Intervalo de Confiança 95%: [{res_mc['ic_95'][0]*100:.2f}%, {res_mc['ic_95'][1]*100:.2f}%]")

    plt.figure(figsize=(8, 4))
    sns.histplot(res_mc['todas_acuracias'], bins=15, kde=True, color='teal')
    plt.title("Distribuição de Acurácias - Monte Carlo CV (100 Iterações)")
    plt.xlabel("Acurácia")
    plt.show()